# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
from section_loads import *

DATA = ROOT / "data"
OUT = ROOT / "outputs"
IMAGES = OUT / "images"
OUT.mkdir(exist_ok=True)

In [ ]:
# Which end of each component to accumulate from. "low" starts at the lowest
# station coordinate, "high" at the highest.
# lh=high, rh=low
STATION_FROM = {"vehicle": "low",
                "fuselage": "low", 
                "wing lh": "high",
                "wing rh": "low",
                "tail lh": "high",
                "tail rh": "low",
                "tail lower": "low"}

In [ ]:
VEHICLE = "vehicle"                    # its sections bin every body node
EXCLUDE_ITEMS = {"point masses"}       # node_ranges items kept out of sections
VEHICLE_CG = np.array([149, 0.0, 100])
INERTIA_SIGN = -1.0
ACCEL_TO_G = 1.0
# ONLY_LOADCASES = ["100245", "10110002"]
ONLY_LOADCASES = None

# Body Load Processing

In [ ]:
sections, node_ranges, load_cases, point_masses = read_tables(DATA)
df, load_cases = read_force_cards(DATA, load_cases, only=ONLY_LOADCASES)
display(load_cases)

df, excluded = apply_exclusions(df, load_cases)
display(excluded)

In [ ]:
check_component_coverage(sections, node_ranges, VEHICLE, EXCLUDE_ITEMS)
check_configurations(sections, load_cases)
 
pd.concat([
    check_card_configurations(df, load_cases),
    check_sections_per_configuration(sections, node_ranges, load_cases,
                                     VEHICLE, EXCLUDE_ITEMS),
])

In [ ]:
df = tag_components(df, node_ranges, EXCLUDE_ITEMS)
node_report(df)

In [ ]:
pm_loads = point_mass_loads(point_masses, load_cases, INERTIA_SIGN, ACCEL_TO_G)
sec, bodies = run_sections(df, sections, load_cases, VEHICLE)
assignment_report(bodies, sections, VEHICLE)

In [ ]:
agree = check_partitions_agree(sec, VEHICLE_CG)
audit = force_audit(df, pm_loads, VEHICLE_CG, sec)
display(audit)

In [ ]:
cg_breakdown(sec, pm_loads, VEHICLE_CG)

In [ ]:
export_section_loads(sec, OUT / "section_loads.csv")
export_point_mass_loads(pm_loads, OUT / "point_mass_loads.csv")
pd.DataFrame({"file": sorted(p.name for p in OUT.glob("*.csv"))})

# Station Envelopes

In [ ]:
diag = station_diagram(sec, direction=STATION_FROM)
station_drivers(diag)

In [ ]:
# plot_all_station_diagrams(diag, out_dir=IMAGES)
plot_all_station_diagrams(diag, out_dir=IMAGES, drivers=5)   # cap the list
# plot_all_station_diagrams(diag, out_dir=IMAGES, envelope=False)